# Phase 3 — Modélisation baseline (XGBoost, FD001)

Objectif de cette phase : obtenir un premier modèle entraîné pour de vrai, avec un RMSE et un score NASA
mesurés sur le jeu de test officiel (`test_FD001.txt` + `RUL_FD001.txt`), reproductibles avec le code
de ce notebook — contrairement aux chiffres de l'ancien notebook exploratoire, supprimé
(voir [ADR-0006](decisions/0006-suppression-baseline-exploratoire.md) et la mise en garde dans le README).

## 1. Imports

On réutilise `src/preprocessing.py`, qui reprend exactement les fonctions déjà validées dans les
notebooks de Phase 1 (chargement, capteurs constants, normalisation, lissage) et de Phase 2
(calcul de la RUL, plafonnement) — pour ne pas recopier une 3e fois cette logique.

In [ ]:
import sys
sys.path.insert(0, "..")  # pour pouvoir importer src/ depuis notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from xgboost import XGBRegressor

from src.preprocessing import (
    load_fd001, get_sensor_columns, find_constant_sensors,
    normalize_minmax, lisser_capteurs, calculer_rul, plafonner_rul, DATA_DIR,
)

FENETRE = 5      # taille de la fenetre glissante, cf. Phase 1 (lissage) et README (benchmark)
PLAFOND_RUL = 125  # plafond de la RUL, cf. Phase 2


## 2. Pipeline Phases 1+2 sur le jeu d'entraînement

On enchaîne les étapes déjà validées : suppression des capteurs constants, normalisation min-max,
lissage par moyenne mobile (moteur par moteur), calcul de la RUL, puis plafonnement à 125 cycles.
Le `scaler` (min-max) est conservé : il faudra l'appliquer tel quel au jeu de test plus tard,
sans le ré-entraîner dessus (sinon fuite d'information entre train et test).

In [ ]:
train = load_fd001("train")  # chargement du fichier d'entrainement FD001

sensor_cols = get_sensor_columns(train.columns)  # les 21 capteurs bruts
constant_sensors = find_constant_sensors(train, sensor_cols)  # capteurs a variance nulle (Phase 1)
train = train.drop(columns=constant_sensors)  # on les retire
sensor_cols_restants = get_sensor_columns(train.columns)  # les 15 capteurs informatifs restants

train, scaler = normalize_minmax(train, sensor_cols_restants)  # normalisation min-max (0 a 1), scaler garde pour le test
train = lisser_capteurs(train, sensor_cols_restants, fenetre=FENETRE)  # moyenne mobile, moteur par moteur
train = calculer_rul(train)  # RUL brute par cycle
train = plafonner_rul(train, plafond=PLAFOND_RUL)  # RUL plafonnee a 125

train.shape  # verification rapide : 20631 lignes, colonnes = 26 - 6 constants + RUL + RUL_cappee = 22


## 3. Features par fenêtre glissante

Une valeur de capteur isolée ne dit pas si le moteur est stable ou en train de se dégrader. On ajoute donc,
pour chaque capteur et chaque cycle, deux nouvelles colonnes calculées sur les 5 derniers cycles
(fenêtre glissante, comme le lissage de la Phase 1, mais ici pour créer des features, pas pour nettoyer le signal) :

- **moyenne glissante** : la tendance récente du capteur.
- **écart-type glissant** : sa dispersion récente — en Phase 2, on a prouvé que cette dispersion double ou
  triple en fin de vie (`sensor_11`) : c'est exactement ce que cette feature permet au modèle de voir.

La fenêtre "glisse" d'un cycle à l'autre : au cycle *t*, elle regarde les cycles *t-4* à *t* (jamais le futur),
et jamais au-delà du début de vie d'un même moteur (calcul refait moteur par moteur, comme le lissage).

In [ ]:
def ajouter_features_glissantes(df, sensor_cols, fenetre):
    """
    Ajoute, pour chaque capteur, deux colonnes calculees sur une fenetre glissante de cycles :
    la moyenne et l'ecart-type des `fenetre` derniers cycles, moteur par moteur.

    Parameters
    ----------
    df : pandas.DataFrame
        Le tableau source (n'est pas modifie sur place, une copie est renvoyee).
    sensor_cols : list de str
        Les noms de colonnes capteurs pour lesquelles calculer les features.
    fenetre : int
        Nombre de cycles utilises pour la fenetre glissante.

    Returns
    -------
    pandas.DataFrame
        Le tableau avec 2 nouvelles colonnes par capteur : "<capteur>_glissante_moy" et "<capteur>_glissante_std".
    """
    df = df.copy()  # copie independante, pour ne pas modifier df sur place

    for sensor_name in sensor_cols:  # on traite chaque capteur, un par un
        colonne_moyenne = sensor_name + "_glissante_moy"
        colonne_ecart_type = sensor_name + "_glissante_std"
        df[colonne_moyenne] = 0.0  # valeurs provisoires, remplacees ligne par ligne juste apres
        df[colonne_ecart_type] = 0.0

        for moteur_id in df["unit_number"].unique():  # on traite chaque moteur separement
            masque = df["unit_number"] == moteur_id  # Vrai sur les lignes de ce moteur, Faux sinon
            # rolling() est causal par defaut (ne regarde que le passe) et ne porte que sur ce moteur
            df.loc[masque, colonne_moyenne] = df.loc[masque, sensor_name].rolling(fenetre, min_periods=1).mean()
            df.loc[masque, colonne_ecart_type] = df.loc[masque, sensor_name].rolling(fenetre, min_periods=1).std()

    return df


train = ajouter_features_glissantes(train, sensor_cols_restants, FENETRE)

# apercu sur le moteur 1 : la valeur brute lissee, sa moyenne glissante et son ecart-type glissant
colonnes_apercu = ["time_cycles", "sensor_11", "sensor_11_glissante_moy", "sensor_11_glissante_std"]
train.loc[train["unit_number"] == 1, colonnes_apercu].head(8)


## 4. Cas particulier : le tout premier cycle de chaque moteur

Au tout premier cycle d'un moteur, la fenêtre glissante ne contient qu'**une seule valeur** — l'écart-type
d'une seule valeur n'existe pas mathématiquement (il faut au moins 2 points pour mesurer une dispersion),
`pandas` renvoie donc `NaN` à cet endroit précis. Ça concerne exactement 1 ligne par moteur par capteur,
soit 100 moteurs × 15 capteurs = 1500 valeurs sur les 20 631 lignes du tableau.

On remplace ces `NaN` par 0 : au tout premier cycle, on n'a encore observé aucune variation, donc une
dispersion de 0 est la valeur la plus honnête qu'on puisse donner au modèle (pas une approximation
hasardeuse, une absence d'information réelle).

In [ ]:
colonnes_ecart_type = [colonne for colonne in train.columns if colonne.endswith("_glissante_std")]

nb_nan_avant = train[colonnes_ecart_type].isna().sum().sum()  # total de NaN sur toutes les colonnes d'ecart-type
train[colonnes_ecart_type] = train[colonnes_ecart_type].fillna(0.0)  # remplace ces NaN par 0
nb_nan_apres = train[colonnes_ecart_type].isna().sum().sum()

print(f"NaN sur les ecarts-types glissants : {nb_nan_avant} avant remplissage, {nb_nan_apres} apres.")
assert nb_nan_avant == 100 * len(sensor_cols_restants)  # 1er cycle de chaque moteur, pour chaque capteur
assert nb_nan_apres == 0


## 5. Construction des features et de la cible

Les features (`X`) sont les 15 capteurs lissés + leurs 15 moyennes glissantes + leurs 15 écarts-types
glissants, soit 45 colonnes. La cible (`y`) est `RUL_cappee` (Phase 2). `groups` retient le numéro de
moteur de chaque ligne — indispensable pour la validation croisée par moteur juste après.

In [ ]:
colonnes_moyenne = [colonne for colonne in train.columns if colonne.endswith("_glissante_moy")]
feature_cols = sensor_cols_restants + colonnes_moyenne + colonnes_ecart_type

X = train[feature_cols]        # les 45 colonnes de features (entrees du modele)
y = train["RUL_cappee"]        # la cible a predire (Phase 2)

# groups sert UNIQUEMENT a GroupKFold plus bas, pour savoir quelles lignes appartiennent au meme moteur.
# scikit-learn ne connait pas le nom "unit_number" : on lui donnera juste cette suite de nombres
# (1,1,1,...,2,2,2,...,100,100), sans aucun contexte. C'est NOUS qui choisissons cette colonne
# precisement parce qu'elle represente "quel moteur" - si on se trompait de colonne ici (par exemple
# time_cycles), le code tournerait quand meme, sans erreur, mais grouperait n'importe quoi.
groups = train["unit_number"]

print(f"Nombre de features : {len(feature_cols)} ({len(sensor_cols_restants)} capteurs lisses + {len(colonnes_moyenne)} moyennes + {len(colonnes_ecart_type)} ecarts-types)")
print(f"X : {X.shape}, y : {y.shape}")

## 6. Le score NASA (Saxena et al., 2008)

Le RMSE traite une surestimation et une sous-estimation de la RUL de façon symétrique. Or le coût réel
ne l'est pas : sous-estimer la RUL entraîne une maintenance un peu prématurée (coût faible), tandis que
la surestimer risque une panne en vol (coût élevé). Le score NASA formalise cette asymétrie :

- si `d = RUL_prédite - RUL_vraie` est **négatif** (sous-estimation, prudent) : pénalité `exp(-d/13) - 1`
- si `d` est **positif** (surestimation, dangereux) : pénalité `exp(d/10) - 1` — croît beaucoup plus vite

Exemple : un écart de 20 cycles coûte `exp(20/13)-1 ≈ 3.66` en sous-estimation, mais `exp(20/10)-1 ≈ 6.39`
en surestimation — near le double, pour le même écart en valeur absolue.

In [ ]:
def score_nasa(y_true, y_pred):
    """
    Calcule le score NASA (Saxena et al., 2008), asymetrique : penalise plus fortement
    une surestimation de la RUL (dangereux) qu'une sous-estimation (prudent).

    Parameters
    ----------
    y_true : array-like
        Les vraies valeurs de RUL.
    y_pred : array-like
        Les valeurs predites par le modele.

    Returns
    -------
    float
        La somme des penalites individuelles sur tous les exemples.
    """
    total = 0.0  # on cumule la penalite de chaque exemple

    for vrai, predit in zip(y_true, y_pred):  # boucle explicite, un exemple a la fois
        d = predit - vrai  # positif si on surestime, negatif si on sous-estime

        if d < 0:  # sous-estimation : penalite plus douce
            total += np.exp(-d / 13) - 1
        else:  # surestimation : penalite plus severe
            total += np.exp(d / 10) - 1

    return total


# sanity check avant utilisation reelle : on verifie la fonction sur un exemple ou le calcul a la main est simple
test_surestimation = score_nasa([100], [120])  # d = +20
test_sousestimation = score_nasa([100], [80])  # d = -20
print(f"surestimation +20 : {test_surestimation:.3f} (attendu {np.exp(2) - 1:.3f})")
print(f"sous-estimation -20 : {test_sousestimation:.3f} (attendu {np.exp(20 / 13) - 1:.3f})")
assert abs(test_surestimation - (np.exp(2) - 1)) < 0.01
assert abs(test_sousestimation - (np.exp(20 / 13) - 1)) < 0.01


## 7. Validation croisée par moteur (GroupKFold)

Si on répartissait les lignes au hasard entre entraînement et validation, des cycles du **même moteur**
pourraient se retrouver des deux côtés. Le modèle apprendrait alors en partie à reconnaître ce moteur
précis plutôt qu'un patron général de dégradation — une fuite d'information qui rendrait le score de
validation artificiellement optimiste, sans rapport avec la performance réelle sur un moteur jamais vu.

`GroupKFold` avec `unit_number` comme groupe garantit qu'un moteur donné va **entièrement** d'un côté
(train) ou de l'autre (validation), jamais coupé en deux. On vérifie ça explicitement à chaque pli.

**`shuffle=True, random_state=42`** : par défaut, `GroupKFold` répartit les moteurs entre les plis selon
l'ordre où ils apparaissent dans le fichier, pas au hasard — si cet ordre correspondait à autre chose
qu'un simple ordre de simulation (par exemple des moteurs triés par robustesse), ça pourrait biaiser
chaque pli sans qu'on le sache. Vérifié : avec `shuffle=True`, le RMSE moyen ne change presque pas
(18.61 sans mélange vs 18.64 avec) — donc pas de biais cachés ici — mais on garde le mélange (avec une
graine fixe pour rester reproductible) pour ne plus jamais avoir à supposer que l'ordre du fichier n'a
pas d'importance.

In [ ]:
# gkf.split(...) est une fonction de la bibliotheque scikit-learn (pas notre code) : c'est A L'INTERIEUR
# d'elle, invisible ici, qu'elle regarde les valeurs uniques de "groups" (les 100 moteurs), decide
# comment les repartir en 5 paquets (melanges grace a shuffle=True, random_state=42 pour rester
# reproductible), puis traduit chaque paquet de moteurs en liste de numeros de ligne. On ne voit que
# le resultat (idx_train, idx_val), jamais le mecanisme interne - d'ou la verification juste en dessous.
gkf = GroupKFold(n_splits=5, shuffle=True, random_state=42)
rmse_par_fold = []
nasa_par_fold = []

for i, (idx_train, idx_val) in enumerate(gkf.split(X, y, groups=groups)):  # 5 plis
    # idx_train / idx_val = juste des listes de numeros de ligne pour ce tour precis,
    # differentes a chaque iteration de la boucle
    X_tr, X_val = X.iloc[idx_train], X.iloc[idx_val]
    y_tr, y_val = y.iloc[idx_train], y.iloc[idx_val]

    # verification explicite qu'aucun moteur n'apparait des deux cotes de ce pli : on ne fait pas
    # confiance aveuglement au mecanisme cache de GroupKFold, on verifie le resultat nous-memes
    moteurs_train = set(groups.iloc[idx_train])
    moteurs_val = set(groups.iloc[idx_val])
    assert len(moteurs_train & moteurs_val) == 0, "un moteur est present des deux cotes !"

    # un modele TOUT NEUF a chaque tour (recree dans la boucle) : aucune memoire du tour precedent
    modele = XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42)
    modele.fit(X_tr, y_tr)           # apprend uniquement sur les moteurs de ce tour
    y_pred = modele.predict(X_val)   # predit sur des moteurs jamais vus par ce modele

    # meme calcul de RMSE qu'a la main (racine de la moyenne des ecarts au carre)
    rmse = np.sqrt(np.mean((y_val.values - y_pred) ** 2))
    nasa = score_nasa(y_val.values, y_pred)
    rmse_par_fold.append(rmse)  # on stocke pour faire la moyenne des 5 tours a la fin
    nasa_par_fold.append(nasa)

    print(f"Pli {i + 1} : {len(moteurs_train)} moteurs train / {len(moteurs_val)} moteurs val -> RMSE={rmse:.2f}, score NASA={nasa:.1f}")

print(f"\nMoyenne sur les 5 plis : RMSE={np.mean(rmse_par_fold):.2f}, score NASA={np.mean(nasa_par_fold):.1f}")

## 8. Modèle final, entraîné sur les 100 moteurs

La validation croisée sert à estimer la performance de façon fiable — une fois qu'on lui fait confiance,
on entraîne un dernier modèle sur **l'intégralité** des 100 moteurs d'entraînement (plus de données =
meilleur modèle), pour l'évaluer ensuite sur le vrai jeu de test officiel.

In [ ]:
modele_final = XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42)
modele_final.fit(X, y)
print("Modele final entraine sur les 100 moteurs d'entrainement.")


## 9. Évaluation sur le jeu de test officiel

`test_FD001.txt` contient des trajectoires **tronquées avant la panne** : on ne connaît la RUL réelle
qu'au tout dernier cycle enregistré de chaque moteur test, donnée séparément dans `RUL_FD001.txt`
(100 valeurs, une par moteur). On applique donc le pipeline au jeu de test, mais avec deux règles
strictes pour éviter toute fuite d'information avec le train :

1. **Mêmes capteurs constants retirés** (ceux identifiés sur le train, pas recalculés sur le test).
2. **Même `scaler`, en `transform` seul** (jamais `fit` à nouveau) — sinon le test influencerait
   la normalisation, ce qui n'est pas possible en conditions réelles (on ne voit jamais le test à l'avance).

In [ ]:
test = load_fd001("test")  # chargement du fichier de test FD001
test = test.drop(columns=constant_sensors)  # memes capteurs retires que sur le train

# transform seul (pas fit_transform) : on applique le scaler deja appris sur le train, sans le reajuster
test[sensor_cols_restants] = scaler.transform(test[sensor_cols_restants])

test = lisser_capteurs(test, sensor_cols_restants, fenetre=FENETRE)  # meme lissage, moteur par moteur
test = ajouter_features_glissantes(test, sensor_cols_restants, FENETRE)  # memes features glissantes
test[colonnes_ecart_type] = test[colonnes_ecart_type].fillna(0.0)  # meme traitement du 1er cycle

test.shape  # verification rapide : 13096 lignes (moteurs test, trajectoires plus courtes)


### Dernier cycle enregistré de chaque moteur test

C'est le seul point de chaque moteur pour lequel on a une RUL vraie fournie (`RUL_FD001.txt`) —
on ne peut donc évaluer le modèle qu'à cet instant précis, pas sur toute la trajectoire test.

In [ ]:
# pour chaque moteur, on garde la ligne dont time_cycles est maximal (dernier cycle enregistre)
indices_derniers_cycles = test.groupby("unit_number")["time_cycles"].idxmax()
dernier_cycle_par_moteur = test.loc[indices_derniers_cycles].sort_values("unit_number").reset_index(drop=True)

rul_test_vraie = pd.read_csv(f"{DATA_DIR}/RUL_FD001.txt", sep=r"\s+", header=None, names=["RUL_vraie"])

print(f"Derniers cycles : {dernier_cycle_par_moteur.shape[0]} moteurs, RUL vraies fournies : {rul_test_vraie.shape[0]} valeurs")
assert dernier_cycle_par_moteur.shape[0] == 100
assert rul_test_vraie.shape[0] == 100


In [ ]:
X_test = dernier_cycle_par_moteur[feature_cols]
y_test_vraie = rul_test_vraie["RUL_vraie"].values

y_test_pred = modele_final.predict(X_test)
y_test_pred = np.clip(y_test_pred, a_min=0, a_max=None)  # une RUL negative n'a pas de sens physique

rmse_test = np.sqrt(np.mean((y_test_vraie - y_test_pred) ** 2))
nasa_test = score_nasa(y_test_vraie, y_test_pred)

ecarts = y_test_pred - y_test_vraie  # positif = surestimation (dangereux)
nb_surestimes_20 = int(np.sum(ecarts > 20))  # nombre de moteurs surestimes de plus de 20 cycles

print(f"RMSE test officiel        : {rmse_test:.2f}")
print(f"Score NASA test officiel  : {nasa_test:.1f}")
print(f"Moteurs surestimes de plus de 20 cycles : {nb_surestimes_20}/100")


## 10. Validation visuelle : prédictions vs vraies RUL

Un nuage de points prédiction vs vérité, avec la diagonale idéale (prédiction = vérité) : plus les points
sont proches de la diagonale, meilleur est le modèle. Les points au-dessus de la diagonale sont les
surestimations (le cas dangereux).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

ax.scatter(y_test_vraie, y_test_pred, alpha=0.6, label="moteurs test")
limite = max(y_test_vraie.max(), y_test_pred.max()) + 5
ax.plot([0, limite], [0, limite], linestyle="--", color="black", label="prediction = verite")

ax.set_xlabel("RUL vraie (RUL_FD001.txt)")
ax.set_ylabel("RUL predite (modele final)")
ax.set_title(f"Test officiel FD001 - RMSE={rmse_test:.2f}, score NASA={nasa_test:.1f}, surestimes>20={nb_surestimes_20}/100")
ax.legend()

plt.tight_layout()
plt.show()
